# reduce-op-mean-divide — worked example 1: Mean-reduce as all_reduce(SUM) then /= world_size

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `reduce-op-mean-divide`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

`ReduceOp` has no MEAN; the idiom for averaging across ranks is `all_reduce(SUM)` followed by an in-place divide by `world_size`. The divide must be in-place (`tensor /= n`) so any external reference to the same storage (e.g. `param.grad`) keeps pointing at the averaged values rather than getting rebound to a new tensor.

## Worked solution

We model the collective with a `FakeDist` whose `all_reduce(tensor, op='sum')` writes the summed contribution of all ranks into the tensor. The function `mean_reduce_inplace(tensor, world_size)` first calls `all_reduce` (now the tensor holds the sum across ranks), then does `tensor /= world_size` in place. Using `/=` keeps the same storage, so the caller's reference still sees the result; `tensor = tensor / n` would rebind the local name and silently drop the update. We start each rank's gradient at `[rank+1]*3`, average across three ranks (1,2,3 → mean 2), and print the result plus a check that the storage reference was preserved.

In [ ]:
class FakeDist:
    def __init__(self, contribs):
        self.total = sum(contribs)  # tensor sum across ranks
    def all_reduce(self, tensor, op='sum'):
        tensor.copy_(self.total)


def mean_reduce_inplace(tensor, world_size, dist_module):
    dist_module.all_reduce(tensor, op='sum')
    tensor /= world_size  # in-place: preserves caller's storage reference


# three ranks contribute [1,1,1], [2,2,2], [3,3,3]
contribs = [t.tensor([float(r + 1)] * 3) for r in range(3)]
fd = FakeDist(contribs)
grad = t.tensor([1.0, 1.0, 1.0])  # this rank's local grad
ref = grad
mean_reduce_inplace(grad, 3, fd)
print('mean grad:', grad.tolist())
print('reference preserved:', grad is ref)